In [52]:
from mmsystem import SELECTDIB

import ConnectionConfig as cc
from delta import DeltaTable
from pyspark.sql.functions import col, hour, explode, from_unixtime, unix_timestamp
cc.setupEnvironment()

In [53]:
spark = cc.startLocalCluster("factRide")
spark.getActiveSession()

In [54]:
cc.set_connectionProfile("VeloDB")

#Extract the ride from the database 
df_operational_rides = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

#Extract the locks from the database 
df_operational_locks = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "locks") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "lockId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

#Extract the stations from the database 
df_operational_stations = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver" , cc.get_Property("driver")) \
    .option("dbtable", "stations") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

# Make temporary views for rides, stations and locks
df_operational_rides.createOrReplaceTempView("rides")
df_operational_stations.createOrReplaceTempView("stations")
df_operational_locks.createOrReplaceTempView("locks")


#Join the rides with locks and stations to get the zipcode 
df_rides_with_zip = spark.sql("""
SELECT r.*, s.zipcode
    FROM rides r
    LEFT JOIN locks l ON r.startlockId = l.lockId
    LEFT JOIN stations s ON l.stationId = s.stationId
""")

df_rides_with_zip.createOrReplaceTempView("rides_with_zip")

spark.sql("SELECT * FROM rides_with_zip").show()


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|zipcode|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:00|     1208|         31000|       1821|     2186|   2018|
|     9|(51.2182,4.41238)| (51.209,4.43266)|2015-09-22 00:00:00|2012-09-22 00:00:00|     5298|         71164|         50|     2067|   2000|
|    12|(51.2088,4.40834)|(51.2145,4.44373)|2015-09-22 00:00:00|2012-09-22 00:00:00|      957|         59847|       2039|     3038|   2000|
|    13|(51.1744,4.40334)|(51.2228,4.42426)|2015-09-22 00:00:00|2012-09-22 00:00:00|     5413|          5045|       5619|     2717|   2610|
|    18|(51.2023,4.4

In [55]:
#Extract dimWeather
dim_Weather = spark.read.format("delta").load("spark-warehouse/dimWeather")
dim_Weather.createOrReplaceTempView("dimWeather")

# Load the JSON files from the weather folder
df_weather = spark.read.json("./weather/", multiLine=True)

df_weather = df_weather.withColumn("weather_time", from_unixtime(col("dt")))

# Flatten the weather column (assuming it's an array)
df_weather_flat = df_weather.withColumn("weather", explode(col("weather")))

# Flatten the DataFrame
df_weather_flat = df_weather_flat.select(
    col("zipCode"),
    col("weather.main").alias("weather_main"),
    col("weather_time")
)

df_weather_flat.createOrReplaceTempView("weatherResponses")
weather_data_responses = spark.sql("SELECT * FROM weatherResponses")
weather_data_responses.show()

+-------+------------+-------------------+
|zipCode|weather_main|       weather_time|
+-------+------------+-------------------+
|   2140|      Clouds|2024-10-07 20:58:17|
|   2140|       Clear|2024-10-07 20:58:17|
|   2140|        Rain|2024-10-07 20:58:17|
|   2170|      Clouds|2024-10-07 20:58:17|
|   2170|       Clear|2024-10-07 20:58:17|
|   2170|        Rain|2024-10-07 20:58:17|
|   2600|      Clouds|2024-10-07 20:58:17|
|   2600|       Clear|2024-10-07 20:58:17|
|   2600|        Rain|2024-10-07 20:58:17|
|   2000|      Clouds|2024-10-07 20:58:17|
|   2018|      Clouds|2024-10-07 20:58:17|
|   2020|      Clouds|2024-10-07 20:58:17|
|   2030|      Clouds|2024-10-07 20:58:17|
|   2050|      Clouds|2024-10-07 20:58:17|
|   2060|      Clouds|2024-10-07 20:58:17|
|   2100|      Clouds|2024-10-07 20:58:17|
|   2610|      Clouds|2024-10-07 20:58:17|
|   2660|      Clouds|2024-10-07 20:58:17|
|   2000|       Clear|2024-10-07 20:58:17|
|   2000|        Rain|2023-10-06 23:20:00|
+-------+--

In [56]:
# Extract hour from ride start time and weather timestamp
df_rides_with_zip = df_rides_with_zip.withColumn("ride_hour", hour(col("starttime")))
df_weather_flat = df_weather_flat.withColumn("weather_hour", hour(col("weather_time")))

In [57]:
# Join rides with weather responses based on zipcode and hour
df_rides_weather_joined = df_rides_with_zip.join(
    df_weather_flat,
    (df_rides_with_zip["zipcode"] == df_weather_flat["zipCode"]),
    "left"
)

# If no match, set weather as 'unknown'
df_rides_weather_joined = df_rides_weather_joined.fillna("unknown", subset=["weather_main"])

df_rides_weather_joined.createOrReplaceTempView("rides_weather")

df_rides_weather_joined.show()



+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+---------+-------+------------+-------------------+------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|zipcode|ride_hour|zipCode|weather_main|       weather_time|weather_hour|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+---------+-------+------------+-------------------+------------+
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:00|     1208|         31000|       1821|     2186|   2018|        0|   2018|        Rain|2024-10-07 20:58:17|          20|
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:00|     1208|         31000|       1821|     2186|   2018|        0|   2018|       Clear|2023-10-07 00:

In [61]:
#Join the rides with dimWeather based on weather condition

df_final_rides_with_weather = spark.sql("""
    SELECT r.*, 
        CASE
            WHEN d.weatherSK IS NOT NULL THEN 
                CASE
                    WHEN r.weather_main = 'Clouds' THEN d.weatherSK 
                    WHEN r.weather_main = 'Clear' THEN d.weatherSK
                    WHEN r.weather_main = 'Rain' THEN d.weatherSK
                    ELSE 4  
                END
            ELSE 4  -- Assuming 4 is the code for "weather type unknown"
        END AS weatherSK,
        CASE
            WHEN d.weather_category IS NOT NULL THEN 
                CASE
                    WHEN r.weather_main = 'Clouds' THEN d.weather_category 
                    WHEN r.weather_main = 'Clear' THEN d.weather_category
                    WHEN r.weather_main = 'Rain' THEN d.weather_category
                    ELSE 'unknown'
                END
            ELSE 'unknown'  -- Defaulting to 'unknown' if no matching weather data is found
        END AS weather_category
    FROM rides_weather r
    LEFT JOIN dimWeather d 
    ON (
        (r.weather_main = 'Clouds' AND d.weather_condition = 'Cloudy') OR
        (r.weather_main = 'Clear' AND d.weather_condition = 'Sunny') OR
        (r.weather_main = 'Rain' AND d.weather_condition = 'Rainy')    
    ) 
    where r.weather_hour = r.ride_hour
""")

df_final_rides_with_weather.show(500)


+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+---------+-------+------------+-------------------+------------+---------+----------------+
|rideid|       startpoint|         endpoint|          starttime|            endtime|vehicleid|subscriptionid|startlockid|endlockid|zipcode|ride_hour|zipCode|weather_main|       weather_time|weather_hour|weatherSK|weather_category|
+------+-----------------+-----------------+-------------------+-------------------+---------+--------------+-----------+---------+-------+---------+-------+------------+-------------------+------------+---------+----------------+
|     4|(51.2023,4.41208)|(51.2119,4.39894)|2015-09-22 00:00:00|2012-09-22 00:00:00|     1208|         31000|       1821|     2186|   2018|        0|   2018|       Clear|2023-10-07 00:20:00|           0|        1|        pleasant|
|    11|(51.2179,4.41777)| (51.211,4.40724)|2015-09-22 00:00:00|2012-09-22 0

In [60]:
# spark.sql("SELECT weatherSK, temperature_condition, weather_condition, weather_category FROM dimWeather").show()
df_rides_with_zip.select("ride_hour").distinct().show()
df_weather_flat.select("weather_hour").distinct().show()

+---------+
|ride_hour|
+---------+
|        0|
|        8|
|        9|
|       10|
|       12|
|       11|
|       14|
|       18|
|       13|
|       15|
|       16|
|       17|
|       19|
|       20|
|       23|
|        6|
|       22|
|        2|
|        4|
|        5|
+---------+
only showing top 20 rows

+------------+
|weather_hour|
+------------+
|          20|
|          23|
|           0|
+------------+

